# 03 — Stamp Segmentation by Contour Analysis

This notebook implements the **contour-based stamp candidate detection** that is used in the final pipeline (notebooks 04 and 05). Unlike the Hough Circle Transform in `03_geometric_localization.ipynb`, this approach scores contours by **circularity, compactness, and extent** — making it more robust to partially occluded or slightly irregular stamps.

**Input:** Raw scanned document images  
**Output:** Ranked list of stamp ROI candidates; `find_stamp_candidates_by_contours` function used by notebook 04

---
## Pipeline Overview
```
Image → Resize → HSV threshold (H+S+V) → Opening → Closing → 
findContours → Score each contour → Rank → Extract ROI
```

---
## Contents
1. Imports
2. Helper: `show_image`
3. Load sample image
4. Resize for display
5. HSV colour thresholding (three-channel: H, S, V)
6. Morphological operations (opening then closing)
7. Find external contours
8. Function: `contour_stamp_score` — score a contour by shape geometry
9. Rank and filter candidates
10. Visualise ranked candidates
11. Extract best ROI with padding
12. Function: `extract_roi_from_candidate`
13. Visualise top-5 ROI candidates
14. Load batch test images
15. Function: `find_stamp_candidates_by_contours` — full pipeline function
16. Batch test on 5 genuine + 5 forged


## 1. Imports

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

## 2. Helper: `show_image`

In [ ]:
def show_image(title, image, cmap=None, figsize=(8, 10)):
    plt.figure(figsize=figsize)
    plt.imshow(image, cmap=cmap)
    plt.title(title)
    plt.axis("off")
    plt.show()

## 3. Load Sample Image

Loads a forged sample image as the working example throughout this notebook.

> ⚠️ Update `sample_path` and `DATASET_ROOT` to your local paths.


In [ ]:
DATASET_ROOT = Path(r"H:\UOR\7th sem\Computer Vision\final_dataset")

sample_path = Path(
    r"H:\UOR\7th sem\Computer Vision\final_dataset\class_1_forged\300dpi-048.png"
)

image_bgr = cv2.imread(str(sample_path))

print("Loaded:", image_bgr is not None)
print("Shape:", image_bgr.shape)

## 4. Resize for Display

Resizes to 900 px width for consistent processing. The original full-resolution image is used when extracting the final ROI (see notebook 04).


In [ ]:
DISPLAY_WIDTH = 900

h, w = image_bgr.shape[:2]
scale = DISPLAY_WIDTH / w
new_h = int(h * scale)

image_resized = cv2.resize(image_bgr, (DISPLAY_WIDTH, new_h))
image_rgb = cv2.cvtColor(image_resized, cv2.COLOR_BGR2RGB)

show_image("Original Image", image_rgb)

## HSV Colour Thresholding

Converts the image to HSV and creates three separate binary masks — one per channel — then combines them using `cv2.bitwise_and`. A pixel must satisfy all three conditions simultaneously to be included in the final mask.

| Channel | Range | Purpose |
|---|---|---|
| Hue | 90 – 170 | Selects blue/violet stamp ink |
| Saturation | 25 – 255 | Excludes grey/black document text which has near zero saturation |
| Value | 30 – 255 | Excludes very dark pixels that may accidentally pass the Hue and Saturation thresholds |

Adding the Value constraint on top of Hue and Saturation produces a cleaner initial mask with fewer false positives compared to using only two channels.

In [ ]:
image_hsv = cv2.cvtColor(image_resized, cv2.COLOR_BGR2HSV)

h = image_hsv[:, :, 0]
s = image_hsv[:, :, 1]
v = image_hsv[:, :, 2]

# Blue/violet hue range
hue_mask = cv2.inRange(h, 90, 170)

# Keep colored pixels, remove grayscale document text
sat_mask = cv2.inRange(s, 25, 255)

# Avoid very dark noise
val_mask = cv2.inRange(v, 30, 255)

mask = cv2.bitwise_and(hue_mask, sat_mask)
mask = cv2.bitwise_and(mask, val_mask)

show_image("Initial HSV Mask", mask, cmap="gray")

## 6. Morphological Operations

Two operations applied in sequence:

| Step | Operation | Kernel | Purpose |
|---|---|---|---|
| 1 | Opening (erosion → dilation) | Ellipse 3×3 | Remove thin text strokes and fine noise |
| 2 | Closing (dilation → erosion) | Ellipse 11×11 | Fill gaps within stamp body |

Note the order here is **open first, close second** — opposite to notebook 02 — because the smaller opening kernel is safer to apply first on this tighter three-channel mask.


In [ ]:
kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))

mask_opened = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)

mask_cleaned = cv2.morphologyEx(mask_opened, cv2.MORPH_CLOSE, kernel_close)

show_image("After Opening", mask_opened, cmap="gray")
show_image("After Opening + Closing", mask_cleaned, cmap="gray")

## Find External Contours

Runs `cv2.findContours` on the cleaned binary mask to find the outlines of all white regions.

- **`RETR_EXTERNAL`** — retrieves only the outermost contours, ignoring any inner contours nested inside a region. This is important because a stamp may have inner details like text or logos that would create inner contours — only the outer boundary of each region is needed here.
- **`CHAIN_APPROX_SIMPLE`** — compresses each contour by storing only the corner points instead of every single point along the edge, saving memory without losing shape information.

The result is a list of contours where each contour represents the outer boundary of one white region in the cleaned mask.

In [ ]:
contours, _ = cv2.findContours(
    mask_cleaned,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

print("Contours found:", len(contours))

## Function — `contour_stamp_score`

Scores a single contour based on how circular and compact it is using four geometric metrics.

| Metric | Formula | What it measures |
|---|---|---|
| `aspect_ratio` | w / h | How close the shape is to a square — a perfect circle gives 1.0 |
| `compactness` | min(w,h) / max(w,h) | Squareness of the bounding box — close to 1.0 for circles, close to 0 for elongated shapes |
| `extent` | area / bbox_area | How much of the bounding box is filled — a filled circle gives ~0.78 |
| `circularity` | 4π × area / perimeter² | Classic shape roundness — a perfect circle gives exactly 1.0, irregular shapes give lower values |

Before scoring, two rejection rules are applied:
- **Area < 200 px²** — too small to be a stamp fragment
- **Compactness < 0.35** — too elongated, likely a text stroke

The final score is a weighted combination of the three most reliable metrics:
- Compactness → 45%
- Circularity → 35%
- Extent → 20%

**Returns:** a dict containing all metrics and position properties, or `None` if the contour is rejected

In [ ]:
def contour_stamp_score(contour):
    area = cv2.contourArea(contour)

    if area <= 0:
        return None

    x, y, w, h = cv2.boundingRect(contour)
    bbox_area = w * h

    if bbox_area <= 0:
        return None

    aspect_ratio = w / h
    compactness = min(w, h) / max(w, h)
    extent = area / bbox_area

    perimeter = cv2.arcLength(contour, True)

    if perimeter <= 0:
        circularity = 0
    else:
        circularity = (4 * np.pi * area) / (perimeter ** 2)

    # Basic rejection rules
    if area < 200:
        return None

    if compactness < 0.35:
        return None

    # Weighted score
    score = (
        0.45 * compactness +
        0.35 * circularity +
        0.20 * extent
    )

    return {
        "contour": contour,
        "x": x,
        "y": y,
        "w": w,
        "h": h,
        "area": area,
        "aspect_ratio": aspect_ratio,
        "compactness": compactness,
        "extent": extent,
        "circularity": circularity,
        "score": score
    }

## Rank and Filter Candidates

Applies `contour_stamp_score` to all contours and discards any that are rejected by the area or compactness rules. The remaining candidates are sorted by score highest first — so the most circular and compact region is always at index 0.

The top 10 candidates are printed with all their metrics so you can inspect the scores and verify that the actual stamp is ranked at or near the top before proceeding to ROI extraction.

In [ ]:
candidates = []

for contour in contours:
    result = contour_stamp_score(contour)

    if result is not None:
        candidates.append(result)

candidates = sorted(
    candidates,
    key=lambda item: item["score"],
    reverse=True
)

print("Valid candidates:", len(candidates))

for idx, c in enumerate(candidates[:10]):
    print(
        f"{idx}: x={c['x']}, y={c['y']}, w={c['w']}, h={c['h']}, "
        f"area={c['area']:.1f}, compact={c['compactness']:.2f}, "
        f"circ={c['circularity']:.2f}, extent={c['extent']:.2f}, "
        f"score={c['score']:.3f}"
    )

## Visualise Ranked Candidates

Draws a green bounding box around each of the top 10 candidates on the resized image, labelled by rank number in blue. 

The stamp should appear with label 0 or 1 — meaning it scored highest by circularity and compactness. If the stamp appears with a higher rank number it indicates the scoring is not working well on this particular image and the threshold parameters may need adjustment.

In [ ]:
output = image_resized.copy()

for idx, c in enumerate(candidates[:10]):
    x, y, w, h = c["x"], c["y"], c["w"], c["h"]

    cv2.rectangle(output, (x, y), (x + w, y + h), (0, 255, 0), 2)

    cv2.putText(
        output,
        str(idx),
        (x, y - 5),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 0, 0),
        2
    )

output_rgb = cv2.cvtColor(output, cv2.COLOR_BGR2RGB)

show_image("Ranked Stamp Contour Candidates", output_rgb)

## Extract Best ROI with Padding

Crops the region around the top ranked candidate (`candidates[0]`) with padding calculated as 60% of the bounding box width and height on each side. For example if the bounding box is 100px wide, 60px of extra padding is added on the left and right.

The padding is necessary because the contour bounding box only covers the detected ink pixels — it may slightly underestimate the full stamp circle if some ink pixels were lost during thresholding or morphological operations. The ROI coordinates and shape are printed for verification.

In [ ]:
best = candidates[0]

x, y, w, h = best["x"], best["y"], best["w"], best["h"]

pad_x = int(w * 0.6)
pad_y = int(h * 0.6)

img_h, img_w = image_resized.shape[:2]

x1 = max(x - pad_x, 0)
y1 = max(y - pad_y, 0)
x2 = min(x + w + pad_x, img_w)
y2 = min(y + h + pad_y, img_h)

roi = image_resized[y1:y2, x1:x2]
roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)

show_image("Extracted Stamp ROI", roi_rgb, figsize=(6, 6))

print("ROI bbox:", x1, y1, x2, y2)
print("ROI shape:", roi.shape)

## Function — `extract_roi_from_candidate`

A reusable version of the ROI extraction from the previous cell. Takes a `pad_factor` parameter (default 0.8) instead of a hardcoded padding value — so the amount of padding can be adjusted for different images without changing the code.

The padded coordinates are clamped to the image boundaries using `max` and `min` to prevent the crop from going outside the image. Both the cropped ROI and the bounding box coordinates are returned.

This function is called repeatedly in the following cells when extracting ROIs for multiple candidates, and again in notebook 04 when processing the full dataset.

In [ ]:
def extract_roi_from_candidate(image, candidate, pad_factor=0.8):
    x, y, w, h = candidate["x"], candidate["y"], candidate["w"], candidate["h"]

    pad_x = int(w * pad_factor)
    pad_y = int(h * pad_factor)

    img_h, img_w = image.shape[:2]

    x1 = max(x - pad_x, 0)
    y1 = max(y - pad_y, 0)
    x2 = min(x + w + pad_x, img_w)
    y2 = min(y + h + pad_y, img_h)

    roi = image[y1:y2, x1:x2]

    return roi, (x1, y1, x2, y2)

## Visualise Top-5 ROI Candidates

Extracts and displays ROI crops for the top 5 candidates with `pad_factor=1.0` to include slightly more context around each region for better visual inspection. The bounding box coordinates and score are printed below each crop.

Used to visually verify that candidate 0 is actually the stamp and not a false detection before committing to it as the final stamp in notebook 04.

In [ ]:
top_k = min(5, len(candidates))

for idx in range(top_k):
    roi, bbox = extract_roi_from_candidate(
        image_resized,
        candidates[idx],
        pad_factor=1.0
    )

    roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(5, 5))
    plt.imshow(roi_rgb)
    plt.title(f"ROI Candidate {idx}")
    plt.axis("off")
    plt.show()

    print("Candidate:", idx)
    print("BBox:", bbox)
    print("Score:", candidates[idx]["score"])

## 14. Load Batch Test Images

Selects 5 genuine and 5 forged images for a batch test of the pipeline.


In [ ]:
test_images = (
    list((DATASET_ROOT / "class_0_genuine").rglob("*.png"))[:5]
    + list((DATASET_ROOT / "class_1_forged").rglob("*.png"))[:5]
)

print("Test images:", len(test_images))

## Function — `find_stamp_candidates_by_contours`

Packages all the steps developed throughout this notebook into a single reusable function. Given a raw document image it runs the full pipeline in one call — resizing, three-channel HSV thresholding, morphological opening (3×3) then closing (11×11), contour detection, scoring, and ranking.

This is the function used directly by notebook 04 to process the entire dataset — instead of repeating all individual steps for every image, notebook 04 simply calls this function once per image.

**Returns:** `(image_resized, mask_cleaned, candidates)`

In [ ]:
def find_stamp_candidates_by_contours(image_bgr, display_width=900):
    h0, w0 = image_bgr.shape[:2]

    scale = display_width / w0
    new_h = int(h0 * scale)

    image_resized = cv2.resize(image_bgr, (display_width, new_h))

    image_hsv = cv2.cvtColor(image_resized, cv2.COLOR_BGR2HSV)

    h = image_hsv[:, :, 0]
    s = image_hsv[:, :, 1]
    v = image_hsv[:, :, 2]

    hue_mask = cv2.inRange(h, 90, 170)
    sat_mask = cv2.inRange(s, 25, 255)
    val_mask = cv2.inRange(v, 30, 255)

    mask = cv2.bitwise_and(hue_mask, sat_mask)
    mask = cv2.bitwise_and(mask, val_mask)

    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))

    mask_opened = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)
    mask_cleaned = cv2.morphologyEx(mask_opened, cv2.MORPH_CLOSE, kernel_close)

    contours, _ = cv2.findContours(
        mask_cleaned,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    candidates = []

    for contour in contours:
        result = contour_stamp_score(contour)

        if result is not None:
            candidates.append(result)

    candidates = sorted(
        candidates,
        key=lambda item: item["score"],
        reverse=True
    )

    return image_resized, mask_cleaned, candidates

## Batch Test — Genuine and Forged Images

Runs `find_stamp_candidates_by_contours` on each test image and displays the top 5 candidates with green bounding boxes and rank labels. The metrics for each candidate are printed below the image.

The purpose is to verify that the detection and ranking works consistently across different stamp impressions, scan resolutions, and forgery types before running the full dataset extraction in notebook 04. If the stamp does not appear as candidate 0 or 1 on several images, the scoring parameters in `contour_stamp_score` may need adjustment.

In [ ]:
for img_path in test_images:
    print("=" * 100)
    print("Testing:", img_path)

    image_bgr = cv2.imread(str(img_path))

    if image_bgr is None:
        print("Could not load image")
        continue

    image_resized, mask_cleaned, candidates = find_stamp_candidates_by_contours(image_bgr)

    print("Candidates:", len(candidates))

    output = image_resized.copy()

    for idx, c in enumerate(candidates[:5]):
        x, y, w, h = c["x"], c["y"], c["w"], c["h"]

        cv2.rectangle(output, (x, y), (x + w, y + h), (0, 255, 0), 2)

        cv2.putText(
            output,
            str(idx),
            (x, y - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255, 0, 0),
            2
        )

    output_rgb = cv2.cvtColor(output, cv2.COLOR_BGR2RGB)

    show_image("Top 5 Contour Candidates", output_rgb)

    for idx, c in enumerate(candidates[:5]):
        print(
            f"{idx}: x={c['x']}, y={c['y']}, w={c['w']}, h={c['h']}, "
            f"score={c['score']:.3f}, circ={c['circularity']:.3f}, "
            f"compact={c['compactness']:.3f}"
        )